# NB02c — EDA: Photo Corpus Inventory

**Phase 2, Notebook 3 of 5.** Forensic inventory of the 46 menu images across two sources: 16 Yelp Open Dataset photos and 30 NYPL archival scans. No model is loaded in this notebook — all analysis uses PIL and pandas only.

The Yelp vs NYPL quality gap is the VLM guardrail finding documented in the horizon scan. This notebook quantifies and visualises it.

| Figure | Description |
|---|---|
| `fig9_photo_quality_matrix.png` | Resolution scatter + file-size distribution by source |
| `fig10_contact_sheet.png` | PIL thumbnail grid: 8 Yelp vs 8 NYPL at equal display size |

Run cells top-to-bottom. Inspect each output before proceeding.

## 1. Environment setup

PIL, pandas, matplotlib — no GPU, no model load.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image

ROOT = Path().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "configs").exists(), f"Cannot find project root from {Path().resolve()}"

YELP_DIR  = ROOT / "data" / "raw" / "menu_images"
NYPL_DIR  = ROOT / "data" / "raw" / "nypl" / "images"
NYPL_RAW  = ROOT / "data" / "raw" / "nypl"
FIG_DIR   = ROOT / "outputs" / "figures" / "nb02c"
FIG_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.dpi": 150, "savefig.bbox": "tight"})

print(f"Yelp image dir : {YELP_DIR}")
print(f"NYPL image dir : {NYPL_DIR}")
print(f"Figures dir    : {FIG_DIR}")

## 2. Build photo inventory

Scans all 46 images with PIL (downloaded resolution, file size). NYPL images are cross-referenced with `MenuPage.csv` and `Menu.csv` to recover original scan dimensions and menu metadata (date, sponsor, place). Yelp images have no canonical original resolution — they are served at Yelp's standard thumbnail size.

Key distinction: NYPL originals in the archive are 2000–6600 px tall; we downloaded the server-scaled 760 px versions. Even so, they carry ~3.7× more bytes than Yelp.

In [ ]:
rows = []

# Yelp —————————————————————————————————————————————————————————————————————————
for biz_dir in sorted(YELP_DIR.iterdir()):
    if not biz_dir.is_dir():
        continue
    for img_path in sorted(biz_dir.glob("*.jpg")):
        with Image.open(img_path) as im:
            w, h = im.size
        rows.append({
            "source":          "yelp",
            "path":            str(img_path),
            "filename":        img_path.name,
            "business_id":     biz_dir.name,
            "page_id":         None,
            "download_w":      w,
            "download_h":      h,
            "size_bytes":      img_path.stat().st_size,
            "orig_w":          float("nan"),
            "orig_h":          float("nan"),
            "date":            None,
            "sponsor":         None,
            "place":           None,
        })

# NYPL — scan downloaded images ————————————————————————————————————————————————
nypl_rows = []
for img_path in sorted(NYPL_DIR.glob("*.jpg")):
    page_id = int(img_path.stem)
    with Image.open(img_path) as im:
        w, h = im.size
    nypl_rows.append({
        "source":          "nypl",
        "path":            str(img_path),
        "filename":        img_path.name,
        "business_id":     None,
        "page_id":         page_id,
        "download_w":      w,
        "download_h":      h,
        "size_bytes":      img_path.stat().st_size,
    })

# Merge NYPL with MenuPage (filenames = MenuPage.id) ———————————————————————————
mp = pd.read_csv(NYPL_RAW / "MenuPage.csv",
                 usecols=["id", "menu_id", "full_height", "full_width"])
m  = pd.read_csv(NYPL_RAW / "Menu.csv",
                 usecols=["id", "date", "sponsor", "place"])

nypl_df = pd.DataFrame(nypl_rows)
nypl_df = nypl_df.merge(mp, left_on="page_id", right_on="id", how="left")
nypl_df = nypl_df.merge(m,  left_on="menu_id", right_on="id",
                         how="left", suffixes=("_page", "_menu"))
nypl_df = nypl_df.rename(columns={"full_width": "orig_w", "full_height": "orig_h"})
nypl_df = nypl_df[["source","path","filename","business_id","page_id",
                    "download_w","download_h","size_bytes",
                    "orig_w","orig_h","date","sponsor","place"]]

# Merge Yelp with restaurants.parquet for name / city / stars ——————————————————
rest = pd.read_parquet(ROOT / "data" / "processed" / "restaurants.parquet",
                       columns=["business_id","name","city","stars","price_range"])
yelp_df = pd.DataFrame(rows).merge(rest, on="business_id", how="left")

inv = pd.concat([yelp_df, nypl_df], ignore_index=True)

# Derived columns ——————————————————————————————————————————————————————————————
inv["size_kb"]      = (inv["size_bytes"] / 1024).round(1)
inv["download_mpx"] = (inv["download_w"] * inv["download_h"] / 1e6).round(3)
inv["orig_mpx"]     = (inv["orig_w"]     * inv["orig_h"]     / 1e6).round(1)
inv["aspect_ratio"] = (inv["download_w"] / inv["download_h"]).round(2)

print(f"Total images : {len(inv)}")
print(f"  Yelp       : {(inv['source'] == 'yelp').sum()}")
print(f"  NYPL       : {(inv['source'] == 'nypl').sum()}")
print()
print("Yelp images:")
cols_yelp = ["name","city","stars","price_range","download_w","download_h","size_kb"]
print(inv.loc[inv["source"]=="yelp", cols_yelp].sort_values("name").to_string(index=False))
print()
print("NYPL images (first 8):")
cols_nypl = ["date","sponsor","place","download_w","download_h","size_kb","orig_w","orig_h"]
print(inv.loc[inv["source"]=="nypl", cols_nypl].head(8).to_string(index=False))

## 3. Inspect inventory

Per-source aggregate statistics. Highlights the quality gap between Yelp thumbnail photos and NYPL archival scans across three dimensions: downloaded pixel area, original pixel area, and file size.

In [ ]:
for src, grp in inv.groupby("source"):
    label = "Yelp (user-submitted)" if src == "yelp" else "NYPL (archival scan)"
    print(f"── {label}  ({len(grp)} images) ──")
    print(f"  Download resolution")
    print(f"    width   : {grp['download_w'].min():.0f} – {grp['download_w'].max():.0f} px  "
          f"(mean {grp['download_w'].mean():.0f})")
    print(f"    height  : {grp['download_h'].min():.0f} – {grp['download_h'].max():.0f} px  "
          f"(mean {grp['download_h'].mean():.0f})")
    print(f"    Mpx     : {grp['download_mpx'].min():.3f} – {grp['download_mpx'].max():.3f}  "
          f"(mean {grp['download_mpx'].mean():.3f})")
    if grp["orig_mpx"].notna().any():
        print(f"  Original scan resolution (NYPL archive)")
        print(f"    width   : {grp['orig_w'].min():.0f} – {grp['orig_w'].max():.0f} px")
        print(f"    height  : {grp['orig_h'].min():.0f} – {grp['orig_h'].max():.0f} px")
        print(f"    Mpx     : {grp['orig_mpx'].min():.1f} – {grp['orig_mpx'].max():.1f}  "
              f"(mean {grp['orig_mpx'].mean():.1f})")
    print(f"  File size")
    print(f"    min     : {grp['size_kb'].min():.1f} KB")
    print(f"    max     : {grp['size_kb'].max():.1f} KB")
    print(f"    mean    : {grp['size_kb'].mean():.1f} KB")
    print()

# Key ratio
yelp_mean_kb = inv.loc[inv["source"] == "yelp",  "size_kb"].mean()
nypl_mean_kb = inv.loc[inv["source"] == "nypl", "size_kb"].mean()
print(f"File-size ratio (NYPL / Yelp mean): {nypl_mean_kb / yelp_mean_kb:.1f}×")

nypl_dated = inv.loc[inv["source"] == "nypl", "date"].dropna()
print(f"NYPL date range: {nypl_dated.min()} → {nypl_dated.max()} "
      f"({len(nypl_dated)}/{(inv['source']=='nypl').sum()} images have dates)")

yelp_mpx_mean  = inv.loc[inv["source"] == "yelp",  "download_mpx"].mean()
nypl_orig_mean = inv.loc[inv["source"] == "nypl", "orig_mpx"].mean()
print(f"Pixel-area ratio original NYPL vs Yelp download: {nypl_orig_mean / yelp_mpx_mean:.0f}×")

## 4. Figure 9 — Photo quality matrix

Two panels in one figure:

- **Left — Resolution scatter:** download width × height per image, coloured by source, bubble area proportional to file size. Shows the two populations are clearly separated in resolution space.
- **Right — File-size distribution:** violin plot of file size (KB) by source. NYPL images carry far more information per file even after server-side downscaling to 760 px height.

The separation between the two clouds is the quantitative basis for the VLM guardrail: Yelp images are too small and noisy for reliable OCR; NYPL scans are adequate for character-level text extraction.

In [ ]:
SOURCE_COLORS = {"yelp": "#E07B39", "nypl": "#3A86C8"}

yelp = inv[inv["source"] == "yelp"]
nypl = inv[inv["source"] == "nypl"]

fig = plt.figure(figsize=(12, 5))
gs  = gridspec.GridSpec(1, 2, width_ratios=[1.6, 1], wspace=0.35)

# Panel A — scatter ————————————————————————————————————————————————————————————
ax_scatter = fig.add_subplot(gs[0])

for src, grp, label in [
    ("yelp", yelp, "Yelp (user-submitted, n=16)"),
    ("nypl", nypl, "NYPL archival scan (n=30)"),
]:
    ax_scatter.scatter(
        grp["download_w"], grp["download_h"],
        s=grp["size_kb"] * 2.0,
        color=SOURCE_COLORS[src],
        alpha=0.65,
        edgecolors="white",
        linewidths=0.5,
        label=label,
        zorder=3,
    )

ax_scatter.set_xlabel("Download width (px)")
ax_scatter.set_ylabel("Download height (px)")
ax_scatter.set_title("A — Resolution by source
(bubble area ∝ file size)")
ax_scatter.legend(fontsize=8, framealpha=0.8)
ax_scatter.axhline(400, color="grey", linestyle=":", linewidth=0.8,
                   label="400 px reference")
ax_scatter.axvline(400, color="grey", linestyle=":", linewidth=0.8)

# Panel B — violin / strip ————————————————————————————————————————————————————
ax_vio = fig.add_subplot(gs[1])

vio_order = [("yelp", "Yelp"), ("nypl", "NYPL")]
for i, (src, tick_label) in enumerate(vio_order):
    vals = inv.loc[inv["source"] == src, "size_kb"].values
    parts = ax_vio.violinplot(vals, positions=[i], widths=0.6, showmedians=True)
    for pc in parts["bodies"]:
        pc.set_facecolor(SOURCE_COLORS[src])
        pc.set_alpha(0.55)
    parts["cmedians"].set_color("black")
    parts["cbars"].set_color("black")
    parts["cmins"].set_color("black")
    parts["cmaxes"].set_color("black")
    ax_vio.scatter([i] * len(vals), vals,
                   color=SOURCE_COLORS[src], s=18, alpha=0.7, zorder=3)

ax_vio.set_xticks([0, 1])
ax_vio.set_xticklabels([lbl for _, lbl in vio_order])
ax_vio.set_ylabel("File size (KB)")
ax_vio.set_title("B — File size distribution
by source")

# Annotate ratio
yelp_med = yelp["size_kb"].median()
nypl_med = nypl["size_kb"].median()
ax_vio.annotate(
    f"NYPL median {nypl_med:.0f} KB
({nypl_med/yelp_med:.1f}× Yelp median {yelp_med:.0f} KB)",
    xy=(0.97, 0.97), xycoords="axes fraction",
    ha="right", va="top", fontsize=7.5,
    bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="grey", alpha=0.8),
)

fig.suptitle("Figure 9 — Menu photo quality matrix: 16 Yelp vs 30 NYPL images",
             fontsize=11, y=1.01)

out = FIG_DIR / "fig9_photo_quality_matrix.png"
fig.savefig(out)
plt.show()
print(f"Saved: {out.relative_to(ROOT)}")

## 5. Figure 10 — Contact sheet

Eight images from each source displayed at the same canvas size (200 px height). Viewing identical display dimensions makes the quality difference visible directly to a human reader — NYPL scans show legible text, Yelp photos show blurred or skewed photos typical of user-submitted smartphone images.

Yelp images selected: 8 with the largest file size (highest information density within the Yelp pool). NYPL images selected: 8 spanning the date range.

In [ ]:
THUMB_H = 200

def make_thumb(path, target_h=THUMB_H):
    img = Image.open(path).convert("RGB")
    w, h = img.size
    new_w = max(1, int(w * target_h / h))
    return img.resize((new_w, target_h), Image.LANCZOS)

# Select 8 Yelp by largest file size ——————————————————————————————————————————
yelp_sel = (
    inv[inv["source"] == "yelp"]
    .sort_values("size_kb", ascending=False)
    .head(8)["path"]
    .tolist()
)

# Select 8 NYPL spread across date range (dated first, then undated) ——————————
nypl_sorted = (
    inv[inv["source"] == "nypl"]
    .sort_values("date", na_position="last")
)
step = max(1, len(nypl_sorted) // 8)
nypl_sel = nypl_sorted.iloc[::step]["path"].tolist()[:8]

n_cols = 8
fig, axes = plt.subplots(2, n_cols, figsize=(n_cols * 2.2, 5.5))
fig.subplots_adjust(hspace=0.05, wspace=0.04)

row_labels = [
    "Yelp  (user-submitted, top-8 by file size)",
    "NYPL  (archival scan, spread across date range)",
]
row_paths = [yelp_sel, nypl_sel]

for row_idx, (label, paths) in enumerate(zip(row_labels, row_paths)):
    for col_idx in range(n_cols):
        ax = axes[row_idx, col_idx]
        if col_idx < len(paths):
            thumb = make_thumb(paths[col_idx])
            ax.imshow(np.array(thumb))
            kb = Path(paths[col_idx]).stat().st_size / 1024
            ax.set_title(f"{kb:.0f} KB", fontsize=7, pad=2)
        ax.axis("off")
    axes[row_idx, 0].set_ylabel(label, fontsize=8, rotation=0,
                                 labelpad=5, ha="right", va="center")

fig.suptitle(
    "Figure 10 — Contact sheet: Yelp (top) vs NYPL (bottom)  "
    f"— all at {THUMB_H} px display height",
    fontsize=10, y=1.01,
)

out = FIG_DIR / "fig10_contact_sheet.png"
fig.savefig(out, dpi=120)
plt.show()
print(f"Saved: {out.relative_to(ROOT)}")
print()
print("Visual check: NYPL scans should show legible printed text.")
print("Yelp photos may show glare, skew, or partial menu views.")

## 6. Summary card — NB02c

Key numbers and data-honesty notes for the horizon scan's data section.

In [ ]:
print("=" * 60)
print("NB02c SUMMARY — Photo corpus inventory")
print("=" * 60)

yelp_grp = inv[inv["source"] == "yelp"]
nypl_grp = inv[inv["source"] == "nypl"]

print(f"\n── Corpus totals ──")
print(f"  Yelp images : {len(yelp_grp)}")
print(f"  NYPL images : {len(nypl_grp)}")
print(f"  Total       : {len(inv)}")

print(f"\n── Download resolution (px) ──")
for label, grp in [("Yelp", yelp_grp), ("NYPL", nypl_grp)]:
    print(f"  {label}  width  {grp['download_w'].min():.0f}–{grp['download_w'].max():.0f}  "
          f"height {grp['download_h'].min():.0f}–{grp['download_h'].max():.0f}  "
          f"mean Mpx {grp['download_mpx'].mean():.3f}")

print(f"\n── Original archive resolution (NYPL only) ──")
print(f"  width  {nypl_grp['orig_w'].min():.0f}–{nypl_grp['orig_w'].max():.0f} px")
print(f"  height {nypl_grp['orig_h'].min():.0f}–{nypl_grp['orig_h'].max():.0f} px")
print(f"  mean Mpx original : {nypl_grp['orig_mpx'].mean():.1f}")
print(f"  mean Mpx Yelp     : {yelp_grp['download_mpx'].mean():.3f}")
print(f"  → NYPL originals are {nypl_grp['orig_mpx'].mean() / yelp_grp['download_mpx'].mean():.0f}× "
      f"larger in pixel area than Yelp downloads")

print(f"\n── File size ──")
print(f"  Yelp mean  : {yelp_grp['size_kb'].mean():.1f} KB  "
      f"(min {yelp_grp['size_kb'].min():.1f}, max {yelp_grp['size_kb'].max():.1f})")
print(f"  NYPL mean  : {nypl_grp['size_kb'].mean():.1f} KB  "
      f"(min {nypl_grp['size_kb'].min():.1f}, max {nypl_grp['size_kb'].max():.1f})")
print(f"  NYPL/Yelp file-size ratio : {nypl_grp['size_kb'].mean() / yelp_grp['size_kb'].mean():.1f}×")

dated = nypl_grp["date"].dropna()
print(f"\n── NYPL metadata ──")
print(f"  Date range : {dated.min()} → {dated.max()}  ({len(dated)}/30 images dated)")
print(f"  All 30 images matched to MenuPage.csv — original scan dimensions confirmed.")

print(f"\n── Data honesty ──")
print("  Yelp menu photos are Yelp's server-scaled thumbnails (not original uploads).")
print("  0.008% of the Yelp segment had 'menu' photo labels — documented corpus gap.")
print("  NYPL scans downloaded at 760 px max height; archive originals are 2–7× taller.")
print("  Both sources are RGB JPEG — no PNG/TIFF lossless images in this corpus.")
print("  Ground truth (MenuItem.csv) is available for NYPL only — not for Yelp images.")

print(f"\n── Figures saved ──")
for f in sorted(FIG_DIR.glob("*.png")):
    print(f"  {f.name}")
print()
print("Next: NB02d_eda_nypl_historical.ipynb")